<a href="https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 361, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 361 (delta 147), reused 93 (delta 93), pack-reused 178 (from 2)
Receiving objects: 100% (361/361), 1.94 MiB | 16.97 MiB/s, done.
Resolving deltas: 100% (202/202), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [7]:
import pandas as pd
import os, json
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
X = df[feature_cols]
y = df['is_declining']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# score the full inventory — trained model, applied at production-style scale
df['risk_score'] = model.predict_proba(X)[:, 1]

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

Actions map to the two dominant model signals (from w05's feature importances:
`impressions_prev_30d` and `impressions_last_30d`, ~37% combined) plus staleness
(`days_since_last_update`, consistent with the paper's Finding #4 freshness pattern).

- **`review_urgent`** — high risk score AND stale (>90 days since update): the paper's
  own "decay cliff" finding (Finding #2) shows health dropping sharply past ~270 days,
  so pairing model risk with staleness targets the pages most likely to be both
  declining and neglected.
- **`review`** — high risk score, but recently updated: likely a genuine performance
  issue unrelated to freshness — needs content/technical review, not just a refresh.
- **`monitor`** — moderate risk score: not urgent, but worth a lighter periodic check.
- **`healthy`** — low risk score: no action needed.

In [8]:
def assign_action(row):
    if row['risk_score'] >= 0.7 and row['days_since_last_update'] > 90:
        return 'review_urgent', 'HIGH_RISK_AND_STALE'
    elif row['risk_score'] >= 0.7:
        return 'review', 'HIGH_RISK_RECENTLY_UPDATED'
    elif row['risk_score'] >= 0.4:
        return 'monitor', 'MODERATE_RISK'
    else:
        return 'healthy', 'LOW_RISK'

df[['action', 'reason_code']] = df.apply(assign_action, axis=1, result_type='expand')

queue = df.sort_values('risk_score', ascending=False)[
    ['content_id', 'client_id', 'risk_score', 'action', 'reason_code',
     'days_since_last_update', 'avg_position', 'ctr', 'content_age_days']
]

print(queue['action'].value_counts())
queue.head(10)

action
healthy          12406
review            9627
review_urgent     5525
monitor           2442
Name: count, dtype: int64


,content_id,client_id,risk_score,action,reason_code,days_since_last_update,avg_position,ctr,content_age_days
28381,content_73bd82df022f,client_7f2253d7e2,1.0,review,HIGH_RISK_RECENTLY_UPDATED,20,25.6,0.12,141
8312,content_eeec737f5df7,client_7f2253d7e2,1.0,review,HIGH_RISK_RECENTLY_UPDATED,20,22.8,0.10,225
8356,content_ea909ac4580c,client_f74efabef1,1.0,review,HIGH_RISK_RECENTLY_UPDATED,8,6.4,0.00,92
13376,content_558003e14599,client_6208ef0f77,1.0,review_urgent,HIGH_RISK_AND_STALE,104,9.0,0.00,223
24165,content_05d21fdc1fff,client_8722616204,1.0,review,HIGH_RISK_RECENTLY_UPDATED,8,7.4,0.00,96
19461,content_3917f17fd47f,client_7f2253d7e2,1.0,review,HIGH_RISK_RECENTLY_UPDATED,20,34.8,0.23,224
880,content_0ae3c180d412,client_6208ef0f77,1.0,review_urgent,HIGH_RISK_AND_STALE,104,23.4,0.00,151
20616,content_870430512868,client_a88a7902cb,1.0,review,HIGH_RISK_RECENTLY_UPDATED,8,18.5,0.00,98
21993,content_95d9f04979b5,client_7f2253d7e2,1.0,review,HIGH_RISK_RECENTLY_UPDATED,20,20.4,0.00,168
29514,content_799c0a43649c,client_3fdba35f04,1.0,review_urgent,HIGH_RISK_AND_STALE,104,16.8,0.12,223


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

**Intended use:** a weekly starting point for an SEO analyst deciding which pages to
review, not an automated action list. `risk_score` is a continuous ranking signal —
sort within a tier by score, and work top-down until review capacity for the week is
used up, rather than treating every flagged page as equally urgent.

**A real limit, named honestly:** `review` + `review_urgent` together cover 15,152 of
30,000 pages (50.5% of inventory) — far more than any team could review weekly. This
reflects the model's precision-over-recall design (per w02) catching a wide net of
plausible risk, not a claim that half the portfolio needs attention this week. Use
`review_urgent` first, then rank `review` by `risk_score` and take only the top slice
your team has capacity for.

**Other limits:**
- Trained on a single historical snapshot — doesn't account for seasonal content or
  recent algorithm changes.
- `risk_score` reflects a 30-day-vs-prior-30-day trend proxy, not certainty about
  future performance (per w06's claim rewrite).
- Client-grouped validation (0.805 precision) means the score should generalize
  reasonably to new clients, but hasn't been tested on genuinely new data collected
  after this snapshot.

In [9]:
flagged_share = (df['action'].isin(['review', 'review_urgent'])).mean()
print(f"share of inventory flagged for review: {flagged_share:.3f}")

# illustrative: what a realistic weekly capacity-limited queue looks like
weekly_capacity = 200  # example — an analyst's realistic weekly review volume
top_n_queue = queue.head(weekly_capacity)
print(f"\nexample: top {weekly_capacity} by risk_score, for one week's actual workload")
top_n_queue['action'].value_counts()


share of inventory flagged for review: 0.505

example: top 200 by risk_score, for one week's actual workload


,count
action,
review,161
review_urgent,39


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

**What a person must check before acting on any flagged page:**
- Confirm the page's traffic volume is meaningful (per w05's error analysis, false
  positives skew toward lower-traffic pages — a `review` flag on a page with under
  ~20 impressions deserves a sanity check before spending real effort on it).
- Check whether the decline coincides with a known external cause (algorithm update,
  seasonal topic, a competitor's new content) — the model has no awareness of causes,
  only the impression trend pattern itself.
- Verify the page still matches its original search intent before rewriting it — a
  page can decline because the query landscape shifted, not because the content
  itself got worse.

**What should NEVER be automated:**
- Auto-publishing content changes based on the score alone.
- Auto-deleting, de-indexing, or unpublishing any page — `risk_score` is a review
  prompt, not a quality verdict.
- Treating the reason code as a diagnosis — `HIGH_RISK_AND_STALE` says the model
  flagged a pattern, not that staleness is confirmed to be the cause.
- Reporting `risk_score` externally (e.g. to a

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



- **Precision drift check:** periodically sample completed `review` actions and check
  whether the pages genuinely showed the predicted decline pattern in the following
  30-day window — if realized precision drifts well below the validated 0.805, that's
  a retrain signal.
- **Flagged-share drift:** if the flagged share (currently 50.5%) shifts sharply up or
  down on new data, the model's calibration may no longer match the current portfolio
  — worth a review before trusting the queue as-is.
- **Data staleness:** this model was trained on one historical snapshot — a retrain
  cadence (e.g. quarterly) keeps the feature distributions from drifting too far from
  what the model learned.
- **Feature availability:** if any input column becomes unavailable or its measurement
  method changes upstream (e.g. GA4 tracking changes), the model needs re-validation
  before continued use.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [12]:
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)

metrics = {
    "model": "RandomForestClassifier",
    "validation_split": "GroupShuffleSplit by client_id",
    "precision": 0.805,
    "recall": 0.900,
    "f1": 0.850,
    "baseline_precision": 0.524,
    "baseline_recall": 0.100,
    "total_inventory": int(len(df)),
    "flagged_share": float(flagged_share),
    "action_counts": df['action'].value_counts().to_dict(),
}
with open('work/outputs/action_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(metrics)

{'model': 'RandomForestClassifier', 'validation_split': 'GroupShuffleSplit by client_id', 'precision': 0.805, 'recall': 0.9, 'f1': 0.85, 'baseline_precision': 0.524, 'baseline_recall': 0.1, 'total_inventory': 30000, 'flagged_share': 0.5050666666666667, 'action_counts': {'healthy': 12406, 'review': 9627, 'review_urgent': 5525, 'monitor': 2442}}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.